## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [9]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 125.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 299.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 295.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 291.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatibl

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

### Downloading and Loading the model

In [2]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [3]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
#uncomment the below snippet of code if the runtime is connected to GPU.
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=36,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [5]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [6]:
input_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(input_1)

'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs'

- The query isn't really answered in this case.
- It also feels like the answer is written very abruptly, there is no fluid narration.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [7]:
input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(input_2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea'

- The response is very easy to understand since no complex technical terms are used.
- The answer is cut-off mid sentence.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [8]:
input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(input_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

- The first half of this question isn't answered at all.
- Again, the response is very easy to understand due to it's simple wording.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [9]:
input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(input_4)

Llama.generate: prefix-match hit


'\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a'

- The answer is very vague in providing recommended treatments, so I feel like it could be more detailed and specific.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [10]:
input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(input_5)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sl"

- The response is more concentrated in looking for signs in the patient than in treatment steps for a fractured leg, which I believe should be more important.

## Question Answering using LLM with Prompt Engineering

In [11]:
system_message = """
You are a knowledgable professional from the medical field who reviews the manuals and is able to provide the appropriate answers from the context.
Answer using professional language, explaining technical concepts to be easily understood. Answer must be concise, but be able to respond to all the questions sufficiently.
User input will have the context required by you to answer user questions.

If the answer is not found in the context, respond "I don't know".
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [12]:
prompt_input_1 = system_message+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
response(prompt_input_1)

Llama.generate: prefix-match hit


"\n\nSepsis is a life-threatening condition that occurs when an infection causes a dysregulated host response, leading to organ dysfunction. In a critical care unit, managing sepsis involves a multidisciplinary approach aimed at early recognition, prompt initiation of appropriate therapy, and ongoing assessment and adjustment of treatment based on the patient's clinical status.\n\nThe following are key components of sepsis management in a critical care unit:\n\n1. Early Recognition: Identify patients with suspected infection and assess for signs of sepsis using the Sequential [Sepsis"

- Doesn't enter into much detail.
- Answer is cut off.

In [64]:
#TUNING
tune_prompt_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(tune_prompt_input_1, max_tokens=500, temperature=0.1)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs: Ensure airway patency, breathing, and circulation. Provide oxygen support as needed, and initiate fluid resuscitation to maintain adequate blood pressure and organ perfusion.\n3. Antibiotics: Administer broad-spectrum antibiotics based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if clinically feasible.\n4. Source control: Identify and address the source of infection, s

- The answer is very detailed and specific, but still gets cut off mid-sentence.
- Using a low temperature made the model generate a structured, step-by-step clinical protocol with focused, precise medical terminology.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [13]:
prompt_input_2 = system_message+"\n"+ "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(prompt_input_2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is an inflammatory condition of the appendix, a small, finger-like structure that extends from the cecum, the first part of the large intestine. The most common symptoms include:\n\n1. Abdominal pain, usually starting around the navel and then shifting to the lower right side of the abdomen.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever and a general feeling of being unwell.\n5. Constipation or diarrhea.\n'

- Last part of the question isn´t answered.
- Answer is very direct and to the point.

In [65]:
#TUNING
tune_prompt_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(tune_prompt_input_2, max_tokens=450, temperature=0.8)

Llama.generate: prefix-match hit


"\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small finger-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but they typically include:\n\n1. Abdominal pain: The pain is usually located in the lower right quadrant of the abdomen and may be constant or come and go. It is often described as a dull ache that eventually becomes sharp and severe.\n2. Loss of appetite: People with appendicitis may lose their appetite and feel nauseous.\n3. Vomiting: Nausea and vomiting are common symptoms of appendicitis, especially in the later stages of the condition.\n4. Fever: A fever is often present, especially if the appendix has ruptured.\n5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.\n\nAppendicitis cannot be cured via medicine alone. Antibiotics may be used to treat any accompanying infection, but they will not resolve

- Elevating the temperature from 0 to 0.8 encouraged the model to output a more descriptive, conversational list rather than a dry summary.
- Response is still cut-off.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [14]:
prompt_input_3 = system_message+"\n"+ "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(prompt_input_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden onset of well-circumscribed, round or oval, smooth, bald patches on the scalp or other areas of the body. The exact cause of alopecia areata is not fully understood, but it's believed to be related to a dysfunction of the immune system that attacks the hair follicles.\n\nEffective treatments for addressing sudden patchy hair loss include:\n\n1. Corticosteroids: These medications can be applied top"

- First part of the question mostly ignored. It's mentioned at the end, but it's cutt off mid-sentence.
- Other than that, very concise answer.

In [66]:
#TUNING
tune_prompt_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(tune_prompt_input_3 , max_tokens=300, temperature=0.2, top_p=0.98, top_k=20)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune condition that affects the hair follicles, leading to hair loss in small, round patches on the scalp or other parts of the body. The exact cause of alopecia areata is not fully understood, but it's believed to be related to an abnormal immune response that attacks the hair follicles.\n\nThere are several treatments and solutions for addressing sudden patchy hair loss:\n\n1. Corticosteroids: Corticosteroids are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system, allowing the hair to grow back. They can be applied topically or taken orally in the form of pills or injections. Topical corticosteroids are often the first line of treatment for alopecia areata.\n2. Minoxidil: Minoxidil is a medication that has been shown to promote hair growth in some people with alopecia areata. It works by increasing blood flow to the hair follicles and stimulating hair growth. Minoxidil

- Treatments/solutions are now adressed.
- Adding Nucleus (top_p=0.98) and Top-K (top_k=20) sampling with a low temperature (0.2) allowed the model to construct a structured, itemized breakdown.
- The answer is still abruptly cutoff mid-sentence.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [15]:
prompt_input_4 = system_message+"\n"+ "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(prompt_input_4)

Llama.generate: prefix-match hit


'\n\nA person with a traumatic brain injury (TBI) may require various interventions and therapies depending on the severity and location of the injury. The primary goal of treatment is to promote recovery, prevent complications, and manage symptoms. Here are some common treatments for TBIs:\n\n1. Medical management: This includes addressing any life-threatening conditions such as airway obstruction, breathing difficulties, or excessive bleeding. Medications may be used to control seizures, reduce intracranial pressure, or manage pain and other symptoms.\n\n2. Surgical intervention: In some'

- The treatments remain vague in this response, although it generally mentions them and what they can be used for.

In [67]:
#TUNING
tune_prompt_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(tune_prompt_input_4, max_tokens=600, temperature=0.8, top_p=0.85)

Llama.generate: prefix-match hit


"\nThe treatment for a person with a brain injury depends on the severity and location of the injury. Here are some common treatments:\n1. Medical care: The primary goal is to stabilize the patient's condition and prevent further damage. This may include surgery, medication to manage symptoms such as swelling or seizures, and intensive care.\n2. Rehabilitation: This includes physical therapy, occupational therapy, speech-language therapy, and cognitive rehabilitation. The goal is to help the patient regain lost skills and functions, and learn new strategies to compensate for any impairments.\n3. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, depression, anxiety, or memory loss.\n4. Assistive devices: Devices such as wheelchairs, walkers, or communication aids can help patients with mobility and communication challenges.\n5. Support groups: Joining a support group can provide emotional and social support for both the patient 

- Raising max_tokens to 600 provided enough capacity to outline an entire comprehensive response.
- The model expanded on specific clinical interventions which had been vague in the un-tuned version.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [16]:
prompt_input_5 = system_message+"\n"+ "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(prompt_input_5)

Llama.generate: prefix-match hit


'\n\nA leg fracture, specifically a diaphyseal or shaft fracture, is a serious injury that requires prompt medical attention. The following are necessary precautions and treatment steps for a person who has sustained such an injury during a hiking trip:\n\n1. Immobilization: The first step in managing a leg fracture is to immobilize the affected limb using a splint, a cast, or a traction device to prevent further damage and promote healing. This will help maintain alignment of the bone fragments and reduce pain.\n\n2. Pain relief: Over-the-counter pain medications'

- I feel like the first two sentences of the response are a little unecessary in the response and take up space.
- Answer is truncated.

In [68]:
#TUNING
tune_prompt_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(tune_prompt_input_5, max_tokens=500, temperature=0.2, top_p=0.90, top_k=30)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it is essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent any movement that could worsen the injury.\n2. Assess the situation: Check for signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you suspect severe bleeding, apply pressure to the wound with a clean cloth to help control the flow of blood.\n3. Call for medical assistance: If possible, call for emergency medical services or arrange for transportation to the nearest hospital. In the meantime, try to keep the person warm and comfortable.\n4. Splint the leg: Use any available materials, such as sticks, branches, or a makeshift sling, to immobilize the fractured leg. Be careful not to apply too much pressure that could cut off circulation.\n5. Provide pain relie

- The tuned version systematically covers the precautions, treatment steps and recovery.
- Using a low temperature with nucleus and top-$k$ sampling produced a clear, highly structured response organized into distinct sub-sections.
- The response was still truncated even with all the additional tokens allowed and it already looks like a lengthy response.

## Data Preparation for RAG

### Loading the Data

In [17]:
# uncomment and run the following lines for Google Colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
medical_pdf_path = "/content/drive/MyDrive/Great Learning/Project 5 Medical Assistant/medical_diagnosis_manual.pdf"

In [19]:
pdf_loader = PyMuPDFLoader(medical_pdf_path)

In [20]:
medical = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [21]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(medical[i].page_content,end="\n")

Page Number : 1
ximenahidalgo2002@gmail.com
PN6E17YJFU
This file is meant for personal use by ximenahidalgo2002@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
ximenahidalgo2002@gmail.com
PN6E17YJFU
This file is meant for personal use by ximenahidalgo2002@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .......................................................................................................

The first five pages of the manual show the table of contents. It is observed that there are 144 chapters in the manual.

#### Checking the number of pages

In [22]:
len(medical)

4114

### Data Chunking

In [23]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 40
)

In [24]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [25]:
len(document_chunks)

8635

In [26]:
document_chunks[0].page_content

'ximenahidalgo2002@gmail.com\nPN6E17YJFU\nThis file is meant for personal use by ximenahidalgo2002@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [27]:
document_chunks[2].page_content

'Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [28]:
document_chunks[3].page_content

"275\nChapter 23. Approach to the Patient With Liver Disease    ...........................................................................................\n294\nChapter 24. Testing for Hepatic & Biliary Disorders    ......................................................................................................\n305\nChapter 25. Drugs & the Liver    ................................................................................................................................................\n308\nChapter 26. Alcoholic Liver Disease    ....................................................................................................................................\n314\nChapter 27. Fibrosis & Cirrhosis    ............................................................................................................................................\n322\nChapter 28. Hepatitis    ........................................................................................................

In [29]:
document_chunks[6503].page_content

"disorders (eg, chronic renal insufficiency, anorexia nervosa) may lead to hypogonadotropic\nhypogonadism, which resolves after recovery from the underlying disorder. Relative hypogonadism is\nbecoming more common among long-term survivors of childhood cancers treated with craniospinal\nirradiation. Chemotherapy with alkylating drugs may lead to testicular damage and relative\nhypogonadism.\nPanhypopituitarism may occur congenitally or anatomically (eg, in septo-optic dysplasia or Dandy-\nWalker malformation), causing deficiency of hypothalamic-releasing factors or pituitary hormones.\nAcquired hypopituitarism may result from tumors, neoplasia, their treatment, vascular disorders, infiltrative\ndisorders (eg, sarcoidosis, Langerhans' cell histiocytosis), infections (eg, encephalitis, meningitis), or\ntrauma. Hypopituitarism in childhood may cause delayed growth, hypothyroidism, diabetes insipidus,\nhypoadrenalism, and lack of sexual development when puberty is expected. Hormone deficie

In [30]:
document_chunks[6504].page_content

'which testicular enlargement is first noted). Typically, stature is usually short during childhood,\nadolescence, or both but ultimately reaches normal range. Growth velocity is nearly normal, and growth\npattern parallels the lower percentile curves of the growth chart; the pubertal growth spurt is delayed.\nWhen skeletal age is plotted on the growth curve, it essentially equals the percentile curve of the genetic\ntarget. Diagnosis is by exclusion of growth hormone deficiency, hypothyroidism, and hypogonadism\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nChapter 285. Endocrine Disorders in Children\n3045\nximenahidalgo2002@gmail.com\nPN6E17YJFU\nThis file is meant for personal use by ximenahidalgo2002@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

As expected, there are some overlaps:  

- The sentence '*Typically, stature is usually short during childhood* ' appears in both of the last chunks.  
- If the chunk_overlap is increased, the overlapping length of the sentence will also increase.

### Embedding

In [31]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipykernel_43852/4198310515.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


In [32]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [33]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

- The embedding model provides a fixed-length vector for any number of chunks.
- This is necessary because we want to compare them for similarity.

### Vector Database

- Creating the folder

In [34]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

- Creating the medical database

In [35]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

- Opening the database

In [36]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_43852/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [37]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

- Searching keywords in the database and returning the most similar 3 chunks within this topic.

In [39]:
vectorstore.similarity_search("Heart Failure",k=3)

[Document(metadata={'subject': '', 'creator': 'Atop CHM to PDF Converter', 'keywords': '', 'file_path': '/content/drive/MyDrive/Great Learning/Project 5 Medical Assistant/medical_diagnosis_manual.pdf', 'trapped': '', 'page': 2277, 'modDate': 'D:20260809225241Z', 'author': '', 'moddate': '2026-08-09T22:52:41+00:00', 'creationdate': '2012-06-15T05:44:40+00:00', 'format': 'PDF 1.7', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationDate': 'D:20120615054440Z', 'source': '/content/drive/MyDrive/Great Learning/Project 5 Medical Assistant/medical_diagnosis_manual.pdf', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'total_pages': 4114}, page_content='Chapter 211. Heart Failure\nIntroduction\n(Congestive Heart Failure) (For heart failure in children, see p. 2947.)\nHeart failure (HF) is a syndrome of ventricular dysfunction. Left ventricular failure causes\nshortness of breath and fatigue, and right ventricular failure causes peripheral and abdominal\nflui

- From the retrieved chunks, we observe that all the chunks are related to the key terms 'Heart Failure'.

### Retriever

In [40]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}  #If the k value is increased, there is a chance that we might find the answer in even more chunks.
)

- The k value is a hyperparameter that we need to tune to get the best context.

In [41]:
rel_docs = retriever.get_relevant_documents("What is congenital heart disease?")
rel_docs

/tmp/ipykernel_43852/3005035992.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What is congenital heart disease?")


[Document(metadata={'format': 'PDF 1.7', 'creator': 'Atop CHM to PDF Converter', 'source': '/content/drive/MyDrive/Great Learning/Project 5 Medical Assistant/medical_diagnosis_manual.pdf', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'total_pages': 4114, 'page': 3119, 'author': '', 'creationDate': 'D:20120615054440Z', 'subject': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'keywords': '', 'moddate': '2026-08-09T22:52:41+00:00', 'modDate': 'D:20260809225241Z', 'trapped': '', 'file_path': '/content/drive/MyDrive/Great Learning/Project 5 Medical Assistant/medical_diagnosis_manual.pdf'}, page_content="Chapter 293. Congenital Cardiovascular Anomalies\nIntroduction\n(See also Ch. 214.)\nCongenital anomalies of the heart and blood vessels arise during the first 10 wk of embryonic\ndevelopment and are present at birth. The incidence is 1/120 live births; estimated risk is 2 to 3% in\nchildren with an affected 

- We can observe that really only 2 of the 3 contain an answer to the query. Specifically, only the first one really shows a definition. The third chunk goes more in depth about symptoms and treatments.

In [42]:
model_output = llm(
      "What is congenital heart disease?",
      max_tokens=200,         # Set max response length
      temperature=0.0         # Set randomness (0.0 for deterministic output)
)

print(model_output['choices'][0]['text'])

Llama.generate: prefix-match hit



Congenital heart disease refers to a group of heart defects that are present at birth. These defects can affect the structure and function of the heart, as well as the blood vessels that carry blood to and from the heart. Congenital heart disease can range from mild to severe, and some forms may not require treatment while others may require surgery or other interventions.

What causes congenital heart disease?
The exact cause of most congenital heart defects is unknown. However, certain factors have been linked to an increased risk of developing these conditions, including genetic factors, environmental exposures, and maternal health during pregnancy. For example, some congenital heart defects may be inherited from one or both parents, while others may be caused by exposure to certain medications or infections during pregnancy.

What are the symptoms of congenital heart disease?
The symptoms of congenital heart disease can vary widely depending on the specific type and


- The above response is somewhat generic and is solely based on the data the model was trained on, rather than the medical manual.

### System and User Prompt Template

- Prompts guide the model to generate accurate responses. Here, we define two parts:

1. The system message describing the assistant's role.
2. A user message template including context and the question.

In [43]:
qna_system_message = """
You are a knowledgable professional from the medical field who reviews the manuals and is able to provide the appropriate answers from the context.
Answer using professional language, explaining technical concepts to be easily understood. Answer must be concise, but be able to respond to all the questions sufficiently.
User input will have the context required by you to answer user questions.

This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [44]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [45]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [46]:
rag_user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(rag_user_input_1,top_k=20) #k value means that it uses the 20 most relevant chunks to generate an answer

Llama.generate: prefix-match hit


'Answer:\nIn a critical care unit, patients with sepsis should be closely monitored due to their unstable hemodynamic status. The following parameters should be monitored frequently: systemic pressure; central venous pressure (CVP), pulmonary artery occlusive pressure (PAOP), or both; pulse oximetry; arterial blood gases (ABGs); blood glucose, lactate, and electrolyte levels; renal function; and possibly sublingual PCO2. Urine output should also be measured to assess renal perfusion.'

- The answer is very concise and it answers exactly what it is asked using very professional and technical terms.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [47]:
rag_user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(rag_user_input_2,top_k=20)

Llama.generate: prefix-match hit


'Appendicitis is a condition characterized by inflammation of the vermiform appendix, which typically presents with abdominal pain, anorexia, and abdominal tenderness. The diagnosis is clinical, often supplemented by CT or ultrasound. Treatment involves surgical removal of the appendix due to the risk of perforation and potential complications such as peritonitis, gangrene, and sepsis. There is no cure for appendicitis via medicine alone.'

- As can be observed in this answer, like the last one, it is very accurate to the question and goes straight to the point.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [48]:
rag_user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(rag_user_input_3,top_k=20)

Llama.generate: prefix-match hit


'Based on the context provided, sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers. The most common areas affected are the scalp and beard. Alopecia areata is characterized by a decrease in telogen-phase hairs and an increased number of broken hairs. Primary hair shaft abnormalities may be observed on microscopic examination of the hair shaft. Scalp biopsy is indicated when alopecia persists and diagnosis is in doubt, as it can differentiate scar'

- The fisrt part of this query isn't really answered at all. This answer mainly just explains what alopecia areata is and what it is characterized by.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [49]:
rag_user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(rag_user_input_4,top_k=20)

Llama.generate: prefix-match hit


'Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, include early intervention by rehabilitation specialists, prevention of secondary disabilities such as pressure ulcers and joint contractures, prevention of pneumonia, and family education. During the acute phase, daily routine care should include measures to prevent contractures, pressure ulcers, and pneumonia, as well as all measures needed to prevent other complications. Patients with severe cognitive dysfunction require extensive cognitive therapy which is often begun immediately after injury and continued for'

- Like the first two queries, this one is answered very clearly and well structured.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [50]:
rag_user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(rag_user_input_5,top_k=20)

Llama.generate: prefix-match hit


'Based on the context provided, the following are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:\n\n1. Seek medical care as soon as possible to assess the severity of the fracture and receive appropriate treatment.\n2. Immobilize the affected leg using a splint or cast to prevent movement and allow the bone to heal properly. This can help minimize complications such as infection, nerve damage, and muscle atrophy.\n3. Elevate the injured leg above heart level for the first 48 hours after suturing to'

- This is a satisfactory response to the query presented. It is very clear and demonstrates the steps to take in the context of the question.

### Fine-tuning

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [51]:
tune_user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(tune_user_input_1, max_tokens=300)

Llama.generate: prefix-match hit


'Answer:\nIn a critical care unit, patients with sepsis should be closely monitored due to their unstable hemodynamic status. The following parameters should be monitored frequently: systemic pressure; central venous pressure (CVP), pulmonary artery occlusive pressure (PAOP), or both; pulse oximetry; arterial blood gases (ABGs); blood glucose, lactate, and electrolyte levels; renal function; and possibly sublingual PCO2. Urine output should also be measured to assess renal perfusion.\n\nFluid resuscitation with 0.9% saline is initiated until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg. Oliguria with hypotension should not deter from vigorous fluid resuscitation, as the quantity of fluid required often exceeds normal blood volume and may reach up to 10 L over 4 to 12 hours. PAOP or echocardiography can help identify limitations in left ventricular function and incipient pulmonary edema due to fluid overload.\n\nIf a patient remains hypotensive despite CVP or PAOP bein

- max_tokens helps limit helps limit the number of tokens used to answer a query.
- In this case, due to the max_tokens, the response of the model was truncated right in the middle of a sentence.
- The fine-tuned model directly generates highly technical medical protocols.
- The structured paragraph brakes and the beginnig starting with "In a critical care unit," indicate that has learned a standardized template or style during the fine-tuning.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [52]:
tune_user_input_2 = " What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(tune_user_input_2, temperature=0.1, max_tokens=450)

Llama.generate: prefix-match hit


'Appendicitis is a condition characterized by inflammation of the vermiform appendix, which typically presents with abdominal pain, anorexia, and abdominal tenderness. The diagnosis is clinical, often supplemented by CT or ultrasound. Treatment involves surgical removal of the appendix due to the risk of perforation and potential complications such as peritonitis, gangrene, and perforation. There is no cure for appendicitis via medicine alone.'

- In this case, increasing the max_tokens gave the model enough room to complete its full thought without getting cut off mid-sentence.
- Using a low temperature resulted in a highly focused, direct, and factual response without hallucinated or irrelevant details.
- The model successfully addressed all sub-questions in the prompt.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [54]:
tune_user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(tune_user_input_3, top_p=0.98, top_k=20, max_tokens=450)

Llama.generate: prefix-match hit


'Based on the context provided, sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers. The most common areas affected are the scalp and beard. Alopecia areata is characterized by a decrease in telogen-phase hairs and an increased number of broken hairs. Primary hair shaft abnormalities may be observed on microscopic examination of the hair shaft. Scalp biopsy is indicated when alopecia persists and diagnosis is in doubt, as it can differentiate scarring from nonscarring forms.\n\nEffective treatments for addressing sudden patchy hair loss include:\n1. Topical corticosteroids: These are anti-inflammatory agents that help reduce inflammation and promote hair regrowth. They can be applied directly to the affected area in the form of creams, lotions, or solutions.\n2. Oral antimalarials: These medications are used to treat lichen planopilaris and chronic cutaneous lupus lesio

- This shows that introducing top_p and top_k sampling allows the model to draw from a broader vocabulary while still maintaining coherent structure. This results in a comprehensive, well-itemized list of treatments.
- Since the answer is more detailed, the response is cut-off again.
- Since the answer begins with "Based on the context provided," it can be assumed that the model is successfully relying on the documentation retrieved and not just on pre-trained knowledge.

#### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [56]:
tune_user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(tune_user_input_4, temperature=0.8, top_p=0.85, max_tokens=600)

Llama.generate: prefix-match hit


"Based on the context provided, early intervention by rehabilitation specialists is essential for maximal functional recovery from a head injury or traumatic brain injury (TBI). Such intervention includes prevention of secondary disabilities like pressure ulcers, joint contractures, and pneumonia. Before starting rehabilitation therapy, patients should be reevaluated to establish a baseline and help prioritize treatment. Patients with severe cognitive dysfunction require extensive cognitive therapy, which is often initiated immediately after the injury and continued for months or years. There's no explicit mention of specific medical treatments or medications in the context."

- The max-tokens being at 600 allowed the answer to finish in it's entirety, but it might be too much text for a response. The user might get lost in the text (although this answer didn't use of all 600 tokens).
- For this query, the combination of a higher temperature and a moderately constrained nucleus produces a more fluid, conversational narrative style rather than a strict, bulleted list.
- Despite the higher temperature, the model explicitly states at the end: "There's no explicit mention of specific medical treatments or medications in the context." This shows the model refrains from hallucinating ungrounded medical data when the retrieved documents lack specific details.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [58]:
tune_user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(tune_user_input_5, temperature=0.2, top_p=0.90, top_k=30, max_tokens=250)

Llama.generate: prefix-match hit


'Based on the context provided, the following are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:\n\n1. Seek medical care at once to assess the severity of the fracture and receive appropriate treatment.\n2. Immobilize the affected leg using a splint or cast to prevent movement and allow the bone to heal properly. A splint allows patients to apply ice and move more compared to complete bed rest, which can cause complications such as deep venous thrombosis and urinary tract infections.\n3. Elevate the injured leg above heart level for the first 48 hours after suturing or immobilization to reduce swelling and promote proper healing.\n4. Use crutches or other assistive devices to help the patient stay off their feet and avoid putting weight on the affected leg during the initial stages of recovery.\n5. Maintain meticulous wound care by keeping the wound clean and dry, applying nonadherent and impermeable dressings, and using ant

- For a detailed multi-step instructions perhaps a higher amount of tokens must be used to avoid the mid-sentence truncation.
- The low temperature produced a clear, logically ordered, step-by-step list covering emergency action, immobilization, elevation, crutch use, and wound care.
- In this case, combining Nucleus and Top-K sampling allowed the model to maintain precise clinical vocabulary without generating repetitive text.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answers generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [79]:
groundedness_rater_system_message = """
Rate if the AI answer is derived ONLY from the context.
Scale: 1 (Not at all), 2 (Limited), 3 (Good), 4 (Mostly), 5 (Completely).

Instructions:
1. Briefly evaluate if the answer uses facts outside the context.
2. Provide a 1-5 score with a 1-sentence reason.
"""

In [80]:
relevance_rater_system_message = """
Rate if the context directly answers the question.
Scale: 1 (Not at all), 2 (Limited), 3 (Good), 4 (Mostly), 5 (Completely).

Instructions:
1. Briefly evaluate if key details are present in the context.
2. Provide a 1-5 score with a 1-sentence reason.
"""

In [81]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [82]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [83]:
ground,rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",max_tokens=290)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 5 (Completely). The answer is derived solely from the context provided in the question and the Merck Manual excerpt.

 4 (Mostly). The context provides detailed information on the monitoring and treatment of sepsis in a critical care unit, including normalization of blood glucose levels and replacement-dose corticosteroids, frequent monitoring of various physiologic parameters, fluid resuscitation with saline, use of vasopressors if necessary, and administration of oxygen. However, it does not explicitly state that these interventions are part of the protocol for managing sepsis in a critical care unit, so the answer is not a complete match to the question.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [84]:
ground,rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",max_tokens=290)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 5 (Completely). The answer is derived solely from the context of the provided text. No external facts are used.

 5 (Completely). The context directly answers the question by providing information about the common symptoms of appendicitis, the fact that medical treatment is not curative, and the standard surgical procedure for treating it.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [85]:
ground,rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",max_tokens=290)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 5 (Completely). The answer is derived solely from the context and accurately summarizes the information provided about alopecia areata and its treatments.

 5 (Completely). The context directly answers the question by providing information about the causes of sudden patchy hair loss (alopecia areata) and effective treatments for this condition, including topical corticosteroids, injections of corticosteroids, systemic corticosteroids, and immunomodulatory agents like minoxidil.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [86]:
ground,rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",max_tokens=290)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 5 (Completely). The answer is derived solely from the context provided in the text.

 4 (Mostly) - The context discusses various treatments for individuals with brain injuries, including cognitive therapy, prevention of secondary disabilities, and immobilization during the acute phase. While it does not provide an exhaustive list of all possible treatments, it does cover several key interventions that can help address temporary or permanent impairments of brain function.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [87]:
ground,rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",max_tokens=290)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 5 (Completely). The answer is derived solely from the context provided in the text.

 I would rate the context as a 5 for answering the question. The context provides detailed information on the necessary precautions and treatment steps for a person who has sustained a leg fracture during hiking, including immobilization, elevation, wound care, antibiotic use, and consideration of complications such as infection and immobility-related issues.


## Actionable Insights and Business Recommendations

- Develop structured answer formats for clinical use. Treatment questions could automatically be presented under headings such as Immediate management, Diagnosis, Treatment, Monitoring, Complications, and Follow-up. This would allow clinicians to scan information faster during time-sensitive situations.
  
- Create an emergency or rapid-response mode. Since one of the business problems is time-sensitive decision-making, the platform could offer a concise mode that retrieves only the highest-priority clinical actions and displays them in an ordered checklist.
  
- Optimize retrieval rather than simply retrieving more documents. Increasing the number of retrieved chunks does not necessarily improve the answer and can introduce irrelevant information or exceed the LLM's context window. The system should experimentally determine the smallest k that consistently retrieves sufficient evidence.
  
- Implement retrieval-quality monitoring. If the retrieved chunks are irrelevant, even a strong LLM will produce a weak RAG answer. Healthcare organizations should therefore evaluate metrics such as retrieval relevance, answer completeness, faithfulness to retrieved evidence, and unsupported claims.
  
- Create specialty-specific knowledge collections. Instead of searching one enormous medical database for every question, information could be separated into areas such as emergency medicine, dermatology, neurology, surgery, infectious disease, and pharmacology. Routing queries to the appropriate collection could improve retrieval precision and speed.
  
- Integrate the assistant directly into clinical workflows. The greatest value would come from incorporating the system into existing hospital platforms, such as clinical knowledge portals or electronic health-record interfaces, so clinicians do not have to leave their normal workflow to search separate databases manually.
  
- Use role-based access and strong data governance. If the platform eventually incorporates patient-specific information, healthcare centers should restrict access based on professional roles, protect sensitive information, maintain audit logs, and ensure compliance with applicable privacy and security requirements.
  
- Evaluate the system with healthcare professionals before deployment. Doctors, nurses, pharmacists, and other domain experts should score generated responses for accuracy, relevance, completeness, usefulness, and potential clinical risk. Automated evaluation alone would be insufficient for a healthcare application.
  
- Track common clinical questions to improve the knowledge platform. Anonymous usage analytics can reveal which conditions, treatments, or protocols clinicians search for most frequently. Administrators can then prioritize those areas when updating documents, designing quick-reference tools, or providing staff training.
  
- Measure business impact using operational and clinical-support KPIs. Healthcare centers should evaluate whether the system reduces time spent searching for information, improves access to appropriate guidelines, decreases repeated manual searches, increases clinician satisfaction, and improves consistency in evidence-based decision support. These measures would help determine whether the RAG platform produces measurable organizational value.

<font size=6 color='blue'>Power Ahead</font>
___